# Monthly performance (Payers)

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
from common_lib.charts import chart_absolute_and_pct_change, chart_actual_vs_predictions, chart_rate_trend, select_repeat_purchase_offset


## 0. Get data

Query DSI activity for the last 400 days (lookback extended to reach the Feb 2025 YoY baseline window). Data is cached to a pickle to avoid re-running the query on every kernel restart.

In [2]:
# hide-output
query_location = './sql/activity.sql'
parameters = {
    'lookback_days': 560,  # needs to reach Feb 8 2025 for the "same time last year" baseline
    'start_date': '2025-01-01',
    'end_date': '2026-08-31',
    'exclude_networks': [],
}

bqc = BigQueryConnector()

cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 4.32 GB when run.
Estimated query cost: $0.03


In [3]:
# hide-output
import os

force_refresh = True  # Set to True to force re-querying the data and updating the pickle file

if not os.path.exists('./data/activity.pkl') or force_refresh:
    data = bqc.get(query=query_location, is_path=True, query_parameters=parameters)
    data.to_pickle('./data/activity.pkl')
else:
    data = pd.read_pickle('./data/activity.pkl')
    print("Pickle already exists, skipping query.")

In [4]:
# hide-output
data

,user_id,dt,dt_week,dt_month,install_dt,install_dt_week,install_dt_month,days_since_install,usd_net_iap_revenue,usd_net_ad_revenue,payer_flag
0,43C73697F85EA630,2025-03-18,2025-03-16,2025-03-01,2023-01-13,2023-01-08,2023-01-01,795,NaN,NaN,0
1,70A7060BA1EE5CE4,2026-08-03,2026-08-02,2026-08-01,2023-02-20,2023-02-19,2023-02-01,1260,NaN,0.481126,0
2,92E0C0B3C3EFE93A,2025-03-02,2025-03-02,2025-03-01,2024-09-18,2024-09-15,2024-09-01,165,NaN,0.127752,0
3,13DA530F2A60B788,2025-11-11,2025-11-09,2025-11-01,2022-12-23,2022-12-18,2022-12-01,1054,NaN,0.051353,0
4,7A3CC21F448B480C,2025-01-07,2025-01-05,2025-01-01,2024-12-31,2024-12-29,2024-12-01,7,NaN,0.165309,0
...,...,...,...,...,...,...,...,...,...,...,...
75731686,872E6A2A9732D150,2026-04-17,2026-04-12,2026-04-01,2025-02-21,2025-02-16,2025-02-01,420,NaN,0.630976,0
75731687,A2F8EDDA911F4727,2025-03-21,2025-03-16,2025-03-01,2024-07-21,2024-07-21,2024-07-01,243,NaN,0.030845,0
75731688,B0D8EDF67B4AD3C3,2026-01-17,2026-01-11,2026-01-01,2026-01-12,2026-01-11,2026-01-01,5,NaN,NaN,0
75731689,BC8733FB6F76DBE5,2025-09-04,2025-08-31,2025-09-01,2022-11-20,2022-11-20,2022-11-01,1019,NaN,0.008882,0


## 1. Payer trend vs DAU trend (2026)

Main metric: **payer % of DAU** (payers ÷ DAU × 100) — rising means payers are eroding slower than the overall active base, falling means faster. Charted with `chart_rate_trend` since day-over-day % change of a rate is too noisy to show trend direction.

The plain DAU/payer absolute-count charts below are kept as supporting context.

In [5]:
# hide-output
data['dt'] = data['dt'].astype('datetime64[ns]')

# Cap every chart in this notebook to ANALYSIS_END_DATE - adjust as needed (e.g. to exclude
# a partial trailing month, which would understate anything computed over it). The one
# exception is the actual-vs-forecast comparison (section 3), which always wants the latest
# data visible regardless of this cap - total_by_dt_dau_full_all below keeps the *uncapped*
# DAU series for that.
ANALYSIS_END_DATE = pd.Timestamp('2026-08-31')
total_by_dt_dau_full_all = data.groupby('dt')['user_id'].nunique().reset_index(name='users')
data = data[data['dt'] <= ANALYSIS_END_DATE]

In [6]:
# hide-output
# Uncapped totals - full population, no tenure filter (see markdown above for why).
total_by_dt_dau_full = data.groupby('dt')['user_id'].nunique().reset_index(name='users')
total_by_dt_payers_full = (
    data[data['payer_flag'] == 1].groupby('dt')['user_id'].nunique().reset_index(name='users')
)

In [7]:
# hide-output
# Payer % of DAU: what share of active users are payers, trended over time. Rising = payers
# eroding slower than the overall base; falling = faster.
payer_pct_df = pd.merge(total_by_dt_dau_full, total_by_dt_payers_full, on='dt', suffixes=('_dau', '_payers'))
payer_pct_df['users'] = payer_pct_df['users_payers'] / payer_pct_df['users_dau'] * 100
payer_pct_df = payer_pct_df[['dt', 'users']]

chart_rate_trend(payer_pct_df, 'Payer % of DAU')
#chart_rate_trend(payer_pct_df, 'Payer % of DAU', freq='W')

## 2. Repeat-purchase rate

### On exact DX

In [8]:
# hide-output
purchases = data.loc[data['payer_flag'] == 1, ['user_id', 'dt']].drop_duplicates()
all_purchase_index = pd.MultiIndex.from_arrays([purchases['user_id'], purchases['dt']])

offsets = [1, 3, 7, 14, 28]
for n in offsets:
    target_index = pd.MultiIndex.from_arrays([purchases['user_id'], purchases['dt'] + pd.Timedelta(days=n)])
    purchases[f'flag_d{n}'] = target_index.isin(all_purchase_index)

repeat_purchase_df = purchases.groupby('dt')[[f'flag_d{n}' for n in offsets]].mean().reset_index()
repeat_purchase_df = repeat_purchase_df.rename(columns={f'flag_d{n}': f'rate_d{n}' for n in offsets})
for n in offsets:
    repeat_purchase_df[f'rate_d{n}'] *= 100

print(f"Purchase rows: {len(purchases)}, unique payers: {purchases['user_id'].nunique()}")

Purchase rows: 2697807, unique payers: 200217


In [9]:
offset = 7  # switch to 1 / 3 / 7 / 14 / 28 to compare a different repeat-purchase window
metric_label = f'Repeat purchase rate (D{offset})'
total_by_dt_repeat = select_repeat_purchase_offset(repeat_purchase_df, offset, max_dt=data['dt'].max())
chart_rate_trend(total_by_dt_repeat, metric_label)  # daily + 7/28-day rolling avg

offset = 28  # switch to 1 / 3 / 7 / 14 / 28 to compare a different repeat-purchase window
metric_label = f'Repeat purchase rate (D{offset})'
total_by_dt_repeat = select_repeat_purchase_offset(repeat_purchase_df, offset, max_dt=data['dt'].max())
chart_rate_trend(total_by_dt_repeat, metric_label)  # daily + 7/28-day rolling avg

### Within DX

In [10]:
# hide-output
# "Within Dx": any purchase in (dt, dt+N] (not just exactly on dt+N). Equivalent to checking
# only the user's *next* purchase after dt - if that one already falls beyond dt+N, no later
# (further-out) purchase could be within the window either, since purchases are sorted.
purchases_sorted = purchases[['user_id', 'dt']].sort_values(['user_id', 'dt'])
purchases_sorted['next_purchase_dt'] = purchases_sorted.groupby('user_id')['dt'].shift(-1)
purchases_sorted['gap_days'] = (purchases_sorted['next_purchase_dt'] - purchases_sorted['dt']).dt.days

for n in offsets:
    purchases_sorted[f'within_d{n}'] = purchases_sorted['gap_days'] <= n

repeat_purchase_within_df = purchases_sorted.groupby('dt')[[f'within_d{n}' for n in offsets]].mean().reset_index()
repeat_purchase_within_df = repeat_purchase_within_df.rename(columns={f'within_d{n}': f'rate_d{n}' for n in offsets})
for n in offsets:
    repeat_purchase_within_df[f'rate_d{n}'] *= 100


In [11]:
offset = 7  # switch to 1 / 3 / 7 / 14 / 28
total_by_dt_within = select_repeat_purchase_offset(repeat_purchase_within_df, offset, max_dt=data['dt'].max())
chart_rate_trend(total_by_dt_within, f'Repeat purchase rate (within D{offset})', freq='D')

offset = 28  # switch to 1 / 3 / 7 / 14 / 28
total_by_dt_within = select_repeat_purchase_offset(repeat_purchase_within_df, offset, max_dt=data['dt'].max())
chart_rate_trend(total_by_dt_within, f'Repeat purchase rate (within D{offset})', freq='D')

## 3. DAU vs static monthly predictions

Actual monthly-avg DAU (computed here from `activity.sql`) vs three "Plan A" prediction vintages (`reference/`): 20260805, 20260826, and 20260518 (UA 350k scenario).

In [12]:
# hide-output
# label, path, name of the DAU column in that particular file (not consistent across files)
prediction_files = [
    ('2026-08-05 (Q3 200K+50K+50K)', './reference/plan a - 200k 202607 50k x2, prd uplifts (updated 20260805)_pl_table.csv', 'DAU'),
    ('2026-08-26 (Q3 200K+50K+50K)', './reference/plan a - 200k 202607 50k x2, prd uplifts (updated 20260826)_pl_table.csv', 'DAU'),
    #('2026-07-01 (1M Q3)','./reference/plan a - 1m 2026q3, yes uplifts (updated 20260701)_pl_table.csv', 'DAU'),
    #('2026-05-18 (1M Q3)', './reference/plan_a.1_-_2026Q3_ua350k (updated 20260518)_pl_table.csv', 'DAU (avg)'),
]

prediction_series = []
for label, path, dau_col in prediction_files:
    pred = pd.read_csv(path, usecols=['Date', dau_col])
    pred['dt'] = pd.to_datetime(pred['Date'])
    pred = pred.rename(columns={dau_col: 'users'})[['dt', 'users']]
    prediction_series.append((label, pred))

# Uses total_by_dt_dau_full_all (uncapped) rather than total_by_dt_dau_full, so partial August
# shows here - the one place in the notebook where that's wanted. Partial months (like this
# one) still understate that month's true average, since mean() only sees the days available.
actual_monthly_dau = (
    total_by_dt_dau_full_all
    .groupby(total_by_dt_dau_full_all['dt'].dt.to_period('M'))['users'].mean()
    .reset_index()
)
actual_monthly_dau['dt'] = actual_monthly_dau['dt'].dt.to_timestamp()

actual_monthly_dau.tail()

,dt,users
15,2026-04-01,90901.200000
16,2026-05-01,83175.483871
17,2026-06-01,78252.100000
18,2026-07-01,73013.354839
19,2026-08-01,63355.703704


In [13]:
chart_actual_vs_predictions(actual_monthly_dau, prediction_series, start='2026-04-01', end='2027-03-31')

In [14]:
export_notebook_html(
    notebook_path='./payers.ipynb',
    output_path='./notebook_output.html',
)

Saved to notebook_output.html


PosixPath('notebook_output.html')